<div dir="rtl">
<center>
  <p>
    <font size="6">
      <b>« تمرین برنامه‌نویسی اول »</b>
    </font>
  </p>

  <p>
    <font size="4">
      یافتن فیلم‌های دارای کمترین و بیشترین میانگین امتیاز با استفاده از الگوریتم MapReduce
    </font>
  </p>

  <p style="margin-top: 15px;">
    <font size="3" color="#102C52">
      <b>نام و نام خانوادگی:</b> علیرضا جهانبخش &nbsp;&nbsp;|&nbsp;&nbsp; <b>شماره دانشجویی:</b> ۴۰۴۴۲۲۲۰۶
    </font>
  </p>

</center>
</div>


<div dir="rtl" style="text-align:right">

در این تمرین با استفاده از **PySpark و عملیات MapReduce روی RDD**، میانگین امتیاز **تمام فیلم‌های دارای رأی** را از داده‌های کاربران محاسبه می‌کنید. سپس برای یافتن فیلم‌های دارای کمترین و بیشترین میانگین، **فقط فیلم‌هایی را در نظر می‌گیرید که حداقل ۵ رأی ثبت‌شده دارند**.

**خروجی مورد انتظار برای فیلم‌های منتخب:** شناسه فیلم، نام فیلم، میانگین امتیاز و تعداد رأی‌های ثبت‌شده. اگر چند فیلم واجد شرایط میانگین کمینه یا بیشینه یکسان داشته باشند، **همه آن‌ها** باید نمایش داده شوند.

> قسمت‌های `TODO` را خودتان تکمیل کنید. مجاز به استفاده از عملیات RDD مانند `map`، `reduceByKey`، `filter` و `mapValues` هستید. برای محاسبه میانگین، از `pandas.groupby` یا `Spark SQL groupBy` استفاده نکنید؛ هدف، طراحی و اجرای الگوریتم MapReduce است.

</div>

<div dir="rtl" style="text-align:right">

## ۱. فایل‌های ورودی و قالب داده‌ها

دو فایل زیر همراه نوت‌بوک در اختیار شما قرار گرفته است:

- `movies.txt`: اطلاعات فیلم‌ها با جداکننده `|`. فیلد اول شناسه فیلم و فیلد دوم نام فیلم است.
- `users.txt`: داده‌های امتیازدهی با جداکننده **Tab**، در قالب `user_id`, `movie_id`, `rating`, `timestamp`.

ستون چهارم زمان ثبت امتیاز است و در این مسئله استفاده نمی‌شود. فایل‌ها سرستون ندارند.

**نکته مهم:** ابتدا میانگین امتیاز **همه فیلم‌های دارای رأی** را بدون اعمال محدودیت بر تعداد رأی محاسبه کنید. **فقط در سؤال د**، برای انتخاب فیلم‌های دارای کمترین و بیشترین میانگین، فیلم‌هایی را در نظر بگیرید که **حداقل ۵ رأی** دارند. اگر چند فیلم واجد شرایط دارای میانگین کمینه یا بیشینه یکسان باشند، همه موارد هم‌رتبه را گزارش کنید.

</div>

<div dir="rtl" style="text-align:right">

## ۲. آماده‌سازی محیط

سلول نصب فقط در صورتی PySpark را نصب می‌کند که در محیط فعلی موجود نباشد. در Google Colab، Runtime معمولی CPU هم برای این تمرین کافی است.

</div>

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])

from pathlib import Path
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("HW1_MapReduce_MovieAverages").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Spark version: 3.5.3


<div dir="rtl" style="text-align:right">

## ۳. بارگذاری داده‌ها

فایل‌های `movies.txt` و `users.txt` را در پوشه جاری نوت‌بوک قرار دهید. اگر در Colab اجرا می‌کنید و فایل‌ها وجود ندارند، پنجره آپلود باز می‌شود؛ هر دو فایل را انتخاب کنید.

</div>

In [2]:
MOVIES_PATH = Path("movies.txt")
RATINGS_PATH = Path("users.txt")

if not (MOVIES_PATH.exists() and RATINGS_PATH.exists()):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Please place movies.txt and users.txt next to this notebook.")
    print("Please upload BOTH movies.txt and users.txt")
    files.upload()

assert MOVIES_PATH.is_file(), "movies.txt is missing"
assert RATINGS_PATH.is_file(), "users.txt is missing"
print("Input files are ready.")

Input files are ready.


<div dir="rtl" style="text-align:right">

### خواندن نام فیلم‌ها

در این بخش فقط نگاشت شناسه فیلم به نام آن ایجاد می‌شود؛ لازم نیست این تابع را تغییر دهید.

</div>

In [3]:
def load_movie_names(path=MOVIES_PATH):
    movie_names = {}
    with open(path, "r", encoding="ISO-8859-1") as f:
        for line_no, line in enumerate(f, start=1):
            fields = line.rstrip("\n").split("|")
            if len(fields) < 2:
                raise ValueError(f"Invalid movie record at line {line_no}")
            movie_names[int(fields[0])] = fields[1]
    return movie_names

movie_names = load_movie_names()
ratings_lines = sc.textFile(str(RATINGS_PATH.resolve()))

print("Movies with metadata:", len(movie_names))
print("Rating records:", ratings_lines.count())
print("First three rating lines:", ratings_lines.take(3))

Movies with metadata: 1682
Rating records: 100000
First three rating lines: ['196\t242\t3\t881250949', '186\t302\t3\t891717742', '22\t377\t1\t878887116']


<div dir="rtl" style="text-align:right">

## ۴. سؤال الف: طراحی مرحله Map

هر رکورد امتیازدهی را به یک جفت کلید-مقدار تبدیل کنید. **کلید شناسه فیلم** و مقدار، اطلاعات لازم برای به‌دست‌آوردن «مجموع امتیازها» و «تعداد رأی‌ها» باشد.

۱. تابع `parse_rating` را کامل کنید.  
۲. عملیات `map` را روی RDD ورودی اجرا کنید.  
۳. حداقل سه جفت کلید-مقدار خروجی را نشان دهید.

</div>

<div dir="rtl" style="text-align:right">

### طراحی `parse_rating` — رویکرد انتخابی

تابع با دو استراتژی پارس پیاده‌سازی شده است:

1. **پارس اصلی (Tab-separated):** ابتدا رکورد را با `\t` می‌شکند. این حالت استاندارد فرمت `users.txt` است.
2. **Fallback (فاصله سفید):** اگر تعداد فیلدها پس از جداسازی با Tab کمتر از ۳ باشد، تلاش می‌شود با هر فاصله سفیدی (whitespace) جدا شود تا در برابر تغییرات احتمالی فرمت مقاوم‌تر باشد.

همچنین بازه امتیازها بین ۰ تا ۵ در نظر گرفته شده تا مقدار ۰ هم (در صورت وجود در داده‌ها) حذف نشود، برخلاف بازه محدودتر ۱ تا ۵.

</div>


In [4]:
def parse_rating(line):
    """Return (movie_id, (rating_sum, rating_count)) for one rating line."""
    parts = line.strip().split("\t")
    if len(parts) < 3:
        parts = line.strip().split()
    if len(parts) < 3:
        raise ValueError(f"Invalid record format: {line}")
    movie_id = int(parts[1])
    rating = float(parts[2])
    if rating < 0 or rating > 5:
        raise ValueError(f"Rating out of bounds: {rating}")
    return (movie_id, (rating, 1))

mapped_rdd = ratings_lines.map(parse_rating)
print("Map output sample:", mapped_rdd.take(3))


Map output sample: [(242, (3, 1)), (302, (3, 1)), (377, (1, 1))]


<div dir="rtl" style="text-align:right">

## ۵. سؤال ب: طراحی مرحله Reduce

برای هر فیلم، جمع امتیازها و تعداد امتیازهای آن را با **`reduceByKey`** محاسبه کنید.

۱. تابع `combine_stats` را طوری تکمیل کنید که دو مقدار جزئی متعلق به یک کلید را ادغام کند.  
۲. عملیات Reduce را روی خروجی Map اجرا کنید.  
۳. برای چند فیلم نمونه، شناسه، مجموع امتیاز و تعداد رأی را چاپ کنید.

</div>

In [5]:
def combine_stats(left, right):
    """Merge two (rating_sum, rating_count) tuples."""
    sum_ratings = left[0] + right[0]
    count_ratings = left[1] + right[1]
    return (sum_ratings, count_ratings)

stats_by_movie_rdd = mapped_rdd.reduceByKey(combine_stats)
print("Reduce output sample:", stats_by_movie_rdd.take(5))


Reduce output sample: [(242, (467, 117)), (302, (1236, 297)), (346, (459, 126)), (474, (825, 194)), (86, (591, 150))]


<div dir="rtl" style="text-align:right">

## ۶. سؤال ج: محاسبه میانگین فیلم‌ها

از RDD مرحله قبل، یک RDD با قالب زیر بسازید:

`(movie_id, (average_rating, rating_count))`

میانگین برابر مجموع امتیازها تقسیم بر تعداد رأی‌هاست. خروجی پنج فیلم نمونه را نمایش دهید.

</div>

In [6]:
averages_rdd = stats_by_movie_rdd.mapValues(
    lambda stat: (stat[0] / stat[1], stat[1])
)

print("Average output sample:", averages_rdd.take(5))


Average output sample: [(242, (3.9914529914529915, 117)), (302, (4.161616161616162, 297)), (346, (3.642857142857143, 126)), (474, (4.252577319587629, 194)), (86, (3.94, 150))]


<div dir="rtl" style="text-align:right">

## ۷. سؤال د: یافتن فیلم‌های با کمترین و بیشترین میانگین

ابتدا از RDD میانگین‌های محاسبه‌شده در سؤال ج، **فقط فیلم‌هایی را نگه دارید که حداقل ۵ رأی ثبت‌شده دارند**. سپس در میان همین فیلم‌ها، **کمترین و بیشترین میانگین امتیاز** را تعیین کرده و **همه فیلم‌هایی** را که میانگین آن‌ها برابر مقدار کمینه یا بیشینه است، بازیابی کنید.

- خروجی هر ردیف: `movie_id`, `movie_name`, `average_rating`, `rating_count`.
- نام فیلم را از دیکشنری `movie_names` به دست آورید.
- نتیجه را بر اساس شناسه فیلم مرتب کنید تا خروجی هر بار یکسان باشد.
- فقط نتایج نهایی را به درایور بیاورید؛ تمام رکوردهای خام امتیازدهی را `collect` نکنید.

**توجه:** شرط حداقل ۵ رأی فقط برای انتخاب فیلم‌های کمینه و بیشینه در این سؤال است؛ میانگین تمام فیلم‌های دارای رأی باید در سؤال ج محاسبه شده باشد.

</div>

In [7]:
eligible_averages_rdd = averages_rdd.filter(lambda item: item[1][1] >= 5)

min_average = eligible_averages_rdd.map(lambda item: item[1][0]).min()
max_average = eligible_averages_rdd.map(lambda item: item[1][0]).max()

min_movies = eligible_averages_rdd.filter(
    lambda item: abs(item[1][0] - min_average) < 1e-6
).collect()

max_movies = eligible_averages_rdd.filter(
    lambda item: abs(item[1][0] - max_average) < 1e-6
).collect()


Rated movies: 1682
Eligible movies: 1349


<div dir="rtl" style="text-align:right">

### نمایش نتایج
پس از تکمیل سلول قبل، این سلول را اجرا کنید.

</div>

In [8]:
def print_extreme_movies(label, rows, movie_names):
    print(f"\n{label} — {len(rows)} movie(s)")
    print(f"{'movie_id':>8} | {'average':>8} | {'votes':>6} | movie_name")
    print("-" * 80)
    for movie_id, (avg, count) in sorted(rows, key=lambda row: row[0]):
        title = movie_names.get(movie_id, "[UNKNOWN MOVIE]")
        print(f"{movie_id:>8} | {avg:>8.4f} | {count:>6} | {title}")

print_extreme_movies("Minimum average", min_movies, movie_names)
print_extreme_movies("Maximum average", max_movies, movie_names)


Minimum average — 3 movie(s)
movie_id |  average |  votes | movie_name
--------------------------------------------------------------------------------
     314 |   1.0000 |      5 | 3 Ninjas: High Noon At Mega Mountain (1998)
     437 |   1.0000 |      5 | Amityville 1992: It's About Time (1992)
     439 |   1.0000 |      5 | Amityville: A New Generation (1993)

Maximum average — 1 movie(s)
movie_id |  average |  votes | movie_name
--------------------------------------------------------------------------------
    1449 |   4.6250 |      8 | Pather Panchali (1955)


<div dir="rtl" style="text-align:right">

## ۸. سؤال تحلیلی

۱. چرا به‌دست‌آوردن میانگین با `reduceByKey` مبتنی بر **جمع و تعداد**، نسبت به میانگین‌گرفتن مستقیم از میانگین‌های جزئی مطمئن‌تر است؟  
۲. آیا فیلمی که فقط یک امتیاز ۵ گرفته است از نظر میزان استقبال یا کیفیت ارزیابی‌ها لزوماً بهتر از فیلمی با میانگین ۴٫۵ و صد رأی است؟ توضیح دهید.


</div>

<div dir="rtl" style="text-align: right; line-height: 1.8;">

### پاسخ سؤال ۱: مقایسه میانگین‌گیری مبتنی بر (مجموع، تعداد) با میانگین‌گیری مستقیم

به‌دست‌آوردن میانگین با استفاده از زوج `(sum, count)` در `reduceByKey` از لحاظ محاسباتی و تئوری سیستم‌های توزیع‌شده کاملاً دقیق و صحیح است، در حالی که میانگین‌گرفتن از میانگین‌های جزئی پارتیشن‌ها با خطای محاسباتی جدی همراه خواهد بود:

1. **نقض خاصیت شرکت‌پذیری و وزن‌دهی نابرابر:** میانگین‌های چند زیرمجموعه در صورتی با هم قابل میانگین‌گیری ساده هستند که تعداد اعضای همه گروه‌ها دقیقاً برابر باشد:

<div dir="ltr" align="center">

$$\frac{\bar{x}_1 + \bar{x}_2}{2} \neq \frac{n_1 \bar{x}_1 + n_2 \bar{x}_2}{n_1 + n_2} \quad (\text{if } n_1 \neq n_2)$$

</div>

   در سیستم‌های کلان‌داده، داده‌ها در پارتیشن‌های مختلف با اندازه‌های نابرابر توزیع شده‌اند. اگر مستقیماً از میانگین پارتیشن‌ها میانگین بگیریم، اثر رأی فیلمی در یک پارتیشن کم‌حجم به اندازه رأی صدها فیلم در یک پارتیشن پرتراکم وزن داده می‌شود و میانگین نهایی دچار سوگیری شدید می‌گردد.
2. **حفظ آماره بسنده (Sufficient Statistics):** زوج `(sum, count)` تحت عملگر جمع دارای دو ویژگی حیاتی **شرکت‌پذیری (Associativity)** و **جابه‌جایی‌پذیری (Commutativity)** است. این امر به موتور پردازشی اسپارک اجازه می‌دهد از مکانیزم بهینه‌ساز **Combiner** در سمت مپِر استفاده کند تا تجمیع محلی صورت پذیرد و از ترافیک سنگین شبکه در مرحله **Shuffle** جلوگیری شود.

</div>


<div dir="rtl" style="text-align: right; line-height: 1.8;">

### پاسخ سؤال ۲: تحلیل مقایسه‌ای کیفیت ارزیابی فیلم تک‌رأی در برابر فیلم با آرای بالا

**خیر؛ فیلمی که صرفاً یک امتیاز ۵ دریافت کرده، لزوماً بهتر از فیلمی با میانگین ۴٫۵ و صد رأی نیست.** این امر بر پایه اصول آماری و مهندسی داده به شرح زیر توجیه می‌شود:

1. **واریانس برآورد و قانون اعداد بزرگ (Law of Large Numbers):** برآورد میانگین با یک تک‌نمونه ($N = 1$) دارای واریانس بسیار بالایی است و فاصله اطمینان (Confidence Interval) آن بسیار عریض و نامطمئن است. در نقطه مقابل، میانگین ۴٫۵ با $N = 100$ به دلیل حجم بالای نمونه، از خطای استاندارد ناچیزی برخوردار بوده و نمادی مطمئن از رضایت جمعی و پایدار کاربران است.
2. **سوگیری شروع سرد (Cold-Start & Selection Bias):** ممکن است تک‌امتیاز ۵ توسط عوامل ساخت فیلم یا فردی با سلیقه‌ای خاص ثبت شده باشد. یک امتیاز منفرد هیچ نمودی از اقبال عمومی یا کیفیت اثر بازتاب نمی‌دهد.
3. **استاندارد سامانه‌های توزیع‌شده توصیه‌گر:** در سیستم‌های توصیه‌گر تجاری (نظیر Netflix و IMDb)، برای رفع این ناهنجاری همواره یک حد آستانه حداقل رأی (مانند $\ge 5$ یا $\ge 50$) اعمال می‌شود یا از فرمول‌های رتبه‌بندی وزنی بیزی (Bayesian Weighted Rating) بهره گرفته می‌شود تا از نمایش فیلم‌های تک‌رأی در صدر فهرست بهترین‌ها جلوگیری شود.

</div>


<div dir="rtl" style="text-align:right">

**توجه:** در سؤال ج میانگین همه فیلم‌های دارای رأی محاسبه می‌شود؛ اما در سؤال د، فقط فیلم‌های دارای **حداقل ۵ رأی** در تعیین کمینه و بیشینه شرکت می‌کنند. اگر چند فیلم واجد شرایط میانگین یکسانی در حد کمینه یا بیشینه داشته باشند، **نمایش همه آن‌ها الزامی است** و نمایش تنها یک فیلم کافی نیست.

</div>